# Fase 4.1 — Validación cruzada repetida, agrupada y anidada de la batería de modelos

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

Aquí entreno y valido los modelos de la lista cerrada del protocolo (v1.1, sección 3) con **las mismas particiones** para todos. Son 5 pliegues por cliente, repetidos 20 veces: 100 ajustes por modelo.

En cada uno de los 100 ajustes:

1. el modelo solo ve el entrenamiento externo;
2. sus hiperparámetros se eligen con un 3-fold interno, también agrupado por cliente, dentro de ese entrenamiento;
3. se predice el pliegue de prueba, que el modelo nunca vio.

El resultado es, para cada modelo, una matriz de 20 × 733 predicciones fuera de pliegue. La comparación, la calibración y la regla de selección están en el notebook 10. Este notebook solo produce las predicciones y documenta qué hiperparámetros se eligieron.

Las predicciones llevan una fila por crédito, así que se guardan en `_privado/`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import json
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd

from tesis_v2 import config, contexto, diseno, io_datos, modelos

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170, "display.max_columns", 30, "display.max_colwidth", 80)

D, ctx, ds = contexto.cargar()
part = json.loads((config.PRIVADO / "particiones.json").read_text(encoding="utf-8"))
assert part["cid"] == D["cid"].astype(int).tolist(), "El orden de la población no coincide con las particiones"
F = np.array(part["pliegue_por_repeticion"])
print(f"Población: {len(D)} créditos, {int(D['y'].sum())} eventos | particiones {F.shape} (huella {part['huella']})")

Población: 733 créditos, 23 eventos | particiones (20, 733) (huella f64cf8b90b908fc1)


## 1. Ajuste de la batería

La tabla siguiente resume cada modelo; las definiciones exactas están en la enmienda v1.1.

| Código | Modelo | Qué se elige en el pliegue interno |
|---|---|---|
| M0a | Tasa base del entrenamiento | — |
| M0b | Regla de políticas vigente (*proxy* P01, P15, P16, P20) | — |
| M1 | *Scorecard* WoE + logística | número de intervalos |
| M2 | Logística penalizada (L2 / L1 / *elastic net*) | penalización y C |
| M2F | Logística de Firth | — |
| M3 | Bayesiana con *prior* de Home Credit | escala τ del *prior* |
| M4 | Configuración v1 (apilado, Home Credit = 80 % del peso) | — |
| M5 | *Boosting* monótono poco profundo | profundidad, árboles y tasa |
| M6 | Supervivencia en tiempo discreto | penalización |
| M7 | Objetivos auxiliares 60+/12m y 30+/6m, recalibrados a 90+ | C |

In [2]:
bat = modelos.bateria()
P, elegidos, tiempos = {}, {}, {}
for k, crear in bat.items():
    t = time.time()
    P[k], elegidos[k] = modelos.validar(D, ctx, crear, F, n_jobs=10)
    tiempos[k] = time.time() - t
    assert not np.isnan(P[k]).any()
    print(f"{k:6s} {tiempos[k]:6.1f} s")
np.savez_compressed(config.PRIVADO / "oof_principal.npz", cid=D["cid"].astype(int).to_numpy(), y=D["y"].to_numpy(),
                    grupo=D["grupo_cliente"].astype(int).to_numpy(), **P)

M0a       1.7 s


M0b       0.3 s


M1        7.5 s


M2       20.8 s


M2F       0.4 s


M3        3.3 s


M4        1.1 s


M5       17.7 s


M6        4.0 s


M7_60     4.7 s


M7_30     4.8 s


## 2. Hiperparámetros elegidos en el pliegue interno

Que un hiperparámetro se elija casi siempre igual en los 100 ajustes indica que la elección es estable. Que salte entre valores indica que la grilla es plana: los datos no distinguen entre configuraciones. Con 23 eventos espero lo segundo.

In [3]:
def legible(k, e):
    if e is None:
        return "—"
    if k == "M2" or k.startswith("M7"):
        C, l1 = e
        pen = {0.0: "L2", 1.0: "L1"}.get(l1, f"EN {l1}")
        return f"{pen}, C={C:.3g}"
    if k == "M5":
        return f"prof={e[0]}, árboles={e[1]}, tasa={e[2]}"
    if k == "M3":
        return f"τ={e}"
    if k == "M1":
        return f"{e} intervalos"
    return f"C={e}"


filas = []
for k, lista in elegidos.items():
    cont = Counter(legible(k, e) for e in lista)
    for valor, n in cont.most_common(4):
        filas.append({"modelo": k, "configuracion": valor, "veces_de_100": n})
hp = pd.DataFrame(filas)
io_datos.guardar_resultado(hp, "fase4", "hiperparametros_elegidos")
hp[~hp["modelo"].isin(["M0a", "M0b", "M2F", "M4"])]

,modelo,configuracion,veces_de_100
2,M1,3 intervalos,66
3,M1,4 intervalos,26
4,M1,5 intervalos,8
5,M2,"L2, C=0.0215",33
6,M2,"EN 0.25, C=0.1",22
7,M2,"L2, C=0.1",19
8,M2,"EN 0.5, C=0.1",13
10,M3,τ=0.25,97
11,M3,τ=0.5,3
13,M5,"prof=1, árboles=50, tasa=0.05",62


**Lectura.** La tabla muestra qué tan informativa es la validación interna con tan pocos eventos: cada pliegue interno tiene entre 5 y 6 eventos.

- **M2:** domina la penalización fuerte (C pequeño). El modelo se encoge mucho hacia la tasa base: los datos no sostienen coeficientes grandes.
- **M3:** el τ elegido indica cuánto creyeron los datos propios al *prior* de Home Credit. Si domina τ pequeño, el modelo se apoya en la información externa; si domina τ grande, la ignora.
- **M5:** las configuraciones más simples (pocos árboles, profundidad 1) indican que no hay interacciones detectables.

Ninguna de estas elecciones usó el pliegue de prueba.

In [4]:
resumen = pd.DataFrame({"modelo": list(P), "segundos": [round(tiempos[k], 1) for k in P],
                        "pd_media_oof": [P[k].mean() for k in P], "pd_min": [P[k].min() for k in P],
                        "pd_max": [P[k].max() for k in P]}).round(4)
io_datos.guardar_resultado(resumen, "fase4", "resumen_predicciones_oof")
_ = io_datos.escribir_registro("09_validacion_cruzada_modelos", {
    "modelos": list(P), "repeticiones": int(F.shape[0]), "pliegues": 5, "huella_particiones": part["huella"],
    "salida_privada": "_privado/oof_principal.npz", "segundos_totales": round(sum(tiempos.values()), 1)})
resumen

,modelo,segundos,pd_media_oof,pd_min,pd_max
0,M0a,1.7,0.0322,0.0315,0.0332
1,M0b,0.3,0.0331,0.0150,0.0438
2,M1,7.5,0.0312,0.0001,0.2488
3,M2,20.8,0.0315,0.0010,0.1741
4,M2F,0.4,0.0418,0.0000,0.6182
5,M3,3.3,0.0315,0.0002,0.2589
6,M4,1.1,0.0319,0.0001,0.4689
7,M5,17.7,0.0314,0.0008,0.2835
8,M6,4.0,0.0490,0.0039,0.5158
9,M7_60,4.7,0.0314,0.0002,0.2501


La tasa media predicha fuera de pliegue debe estar cerca de la tasa observada (3,14 %). El rango de predicciones ya dice mucho de cada modelo:

- los modelos muy penalizados (M2, M6) predicen en un rango estrecho alrededor de la tasa base;
- los menos restringidos (M1, M2F) se atreven a predicciones extremas, que la calibración del notebook 10 premiará o castigará.